In [ ]:
import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout,
    SpatialDropout1D, LayerNormalization, Concatenate, MultiHeadAttention
)

# load data
try:
    df = pd.read_csv("/content/Final_Stock_Data_Macro.csv", index_col='Date', parse_dates=True)
except FileNotFoundError:
    df = pd.read_csv("Final_Stock_Data_Macro.csv", index_col='Date', parse_dates=True)

df = df.apply(pd.to_numeric, errors='coerce').ffill().bfill().fillna(0)

# technical indicators & kinematics
df['MA_14'] = df['Close'].rolling(window=14).mean()
df['Smooth_Trend'] = df['MA_14'].rolling(window=5).mean()
df['Trend_Velocity'] = df['Smooth_Trend'].pct_change(1)
df['Trend_Accel'] = df['Trend_Velocity'].diff(1)
df['Trend_Jerk'] = df['Trend_Accel'].diff(1)
df['Dist_Smooth'] = (df['Close'] - df['Smooth_Trend']) / (df['Smooth_Trend'] + 1e-8)
df['Momentum_13D'] = df['Close'].pct_change(13)

# macro stress shocks
if 'EPU' in df.columns and 'GPR' in df.columns:
    df['EPU_Shock'] = df['EPU'] / (df['EPU'].rolling(window=90, min_periods=1).mean() + 1e-8)
    df['GPR_Shock'] = df['GPR'] / (df['GPR'].rolling(window=90, min_periods=1).mean() + 1e-8)
    df['Macro_Crisis_Index'] = (df['EPU_Shock'] + df['GPR_Shock']) / 2.0

# target definition
df['Target_Trend'] = np.where(df['Smooth_Trend'].shift(-1) > df['Smooth_Trend'], 1, 0)
df.dropna(inplace=True)

target_cols = ['Target_Next_Close', 'Target_Direction', 'Target_Trend']
feature_cols = [c for c in df.columns if c not in target_cols]

ratio_cols = [
    'Trend_Velocity', 'Trend_Accel', 'Trend_Jerk', 'Dist_Smooth', 'Momentum_13D',
    'EPU_Shock', 'GPR_Shock', 'Macro_Crisis_Index'
]
other_cols = [c for c in feature_cols if c not in ratio_cols]

# stationarity & scaling
features_stationary = pd.DataFrame(index=df.index)
for c in other_cols:
    r_mean = df[c].rolling(window=30, min_periods=1).mean()
    r_std = df[c].rolling(window=30, min_periods=1).std().replace(0, 1).fillna(1)
    features_stationary[c] = np.clip((df[c] - r_mean) / (r_std + 1e-8), -4.0, 4.0)

for c in ratio_cols:
    if c in df.columns:
        r_std = df[c].rolling(window=30, min_periods=1).std().replace(0, 1).fillna(1)
        features_stationary[c] = np.clip(df[c] / (r_std + 1e-8), -4.0, 4.0)

features_stationary.replace([np.inf, -np.inf], 0, inplace=True)
features_stationary.fillna(0, inplace=True)

# train/test split
targets = df['Target_Trend'].values
split_idx = int(len(features_stationary) * 0.85)

X_train_raw = features_stationary.iloc[:split_idx].values
X_test_raw = features_stationary.iloc[split_idx:].values
y_train_raw = targets[:split_idx]
y_test_raw = targets[split_idx:]

scaler = StandardScaler(with_mean=False)
X_train_scaled = np.clip(scaler.fit_transform(X_train_raw), -4.0, 4.0)
X_test_scaled = np.clip(scaler.transform(X_test_raw), -4.0, 4.0)

# sequence generation
TIME_STEPS = 14
def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps + 1):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps - 1])
    return np.array(Xs), np.array(ys)

X_train, y_train = create_sequences(X_train_scaled, y_train_raw, TIME_STEPS)
X_test, y_test = create_sequences(X_test_scaled, y_test_raw, TIME_STEPS)

In [ ]:
tf.random.set_seed(42)
reg = tf.keras.regularizers.l2(2e-4)

inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))
x = SpatialDropout1D(0.15)(inputs)

x = Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=reg))(x)
x = LayerNormalization()(x)

attn_out = MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.2)(x, x)
x = LayerNormalization()(x + attn_out)

x = Bidirectional(LSTM(32, return_sequences=False, kernel_regularizer=reg))(x)
x = LayerNormalization()(x)

last_step = inputs[:, -1, :]
combined = Concatenate()([x, last_step])

d1 = Dense(128, activation='swish', kernel_regularizer=reg)(combined)
d1 = LayerNormalization()(d1)
d1 = Dropout(0.25)(d1)

d2 = Dense(64, activation='swish', kernel_regularizer=reg)(d1)
d2 = LayerNormalization()(d2)
d2 = Dropout(0.2)(d2)

d3 = Dense(32, activation='gelu', kernel_regularizer=reg)(d2)
d3 = Dropout(0.15)(d3)

final_features = Concatenate()([d3, last_step])
outputs = Dense(1, activation='sigmoid', kernel_regularizer=reg)(final_features)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=0
)

history = model.fit(
    X_train, y_train,
    epochs=60,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

preds_prob = model.predict(X_test, verbose=0)
preds = (preds_prob > 0.5).astype(int)
acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {acc * 100:.2f}%")
print(classification_report(y_test, preds))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

# التحقق من وجود المتغيرات الهامة قبل البدء
required_vars = ['y_test', 'preds', 'preds_prob']
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    if 'model' in globals() and 'X_test' in globals():
        print("Generating predictions using the active model...")
        preds_prob = model.predict(X_test, verbose=0)
        preds = (preds_prob > 0.5).astype(int)
    else:
        raise ValueError(
            "Required variables or trained model are missing. "
            "Please run preprocessing, model building, and training first."
        )

if 'preds' in globals() or (missing_vars and 'preds' in locals() and preds is not None ):
    # إعداد تصميم الرسوم البيانية بروح كاجل (Kaggle Style)
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig_bg_color = '#ffffff'

    # 1. Heatmap (Confusion Matrix)
    plt.figure(figsize=(7, 5), facecolor=fig_bg_color)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Down (0)', 'Up (1)'],
                yticklabels=['Down (0)', 'Up (1)'],
                annot_kws={"size": 14, "weight": "bold"})
    plt.title('Confusion Matrix: Predicted vs Target', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Target Label', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 2. Flow/Timeline Plot
    plt.figure(figsize=(14, 5), facecolor=fig_bg_color)
    sample_limit = min(200, len(y_test))
    plt.plot(y_test[:sample_limit], label='Actual Target', color='#1f77b4', linewidth=2.5, alpha=0.9)
    plt.plot(preds[:sample_limit], label='Predicted Trend', color='#ff7f0e', linestyle='--', linewidth=2, alpha=0.85)
    plt.title('Prediction Flow vs Actual Target (Timeline Comparison)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Test Samples Index (Time Steps)', fontsize=12, fontweight='bold')
    plt.ylabel('Trend Direction (0 or 1)', fontsize=12, fontweight='bold')
    plt.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.tight_layout()
    plt.show()

    # 3. Probability Distribution Flow
    if 'preds_prob' in globals() and preds_prob is not None:
        plt.figure(figsize=(10, 4), facecolor=fig_bg_color)
        sns.histplot(preds_prob.flatten(), bins=50, kde=True, color='#2ca02c', alpha=0.6)
        plt.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold (0.5)')
        plt.title('Prediction Probability Distribution Flow', fontsize=14, fontweight='bold', pad=15)
        plt.xlabel('Predicted Probability', fontsize=12, fontweight='bold')
        plt.ylabel('Frequency / Count', fontsize=12, fontweight='bold')
        plt.legend(loc='upper center', frameon=True, facecolor='white')
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        plt.show()

In [ ]:
import os
import joblib
import numpy as np
import shutil

# Create folder
os.makedirs("StockVision_Artifacts", exist_ok=True)

# Save trained model
model.save("StockVision_Artifacts/stock_model.keras")

# Save preprocessing tools
joblib.dump(
    scaler,
    "StockVision_Artifacts/scaler.pkl"
)

joblib.dump(
    features_stationary.columns.tolist(),
    "StockVision_Artifacts/feature_columns.pkl"
)

# Save training history
joblib.dump(
    history.history,
    "StockVision_Artifacts/history.pkl"
)

# Save evaluation results
test_dates = features_stationary.index[
    split_idx + TIME_STEPS - 1:
].astype(str).to_numpy()

np.savez_compressed(
    "StockVision_Artifacts/evaluation.npz",
    X_test=X_test,
    y_test=y_test,
    preds=preds,
    preds_prob=preds_prob,
    test_dates=test_dates
)

# Copy dataset
shutil.copy(
    "/content/Final_Stock_Data_Macro.csv",
    "StockVision_Artifacts/Final_Stock_Data_Macro.csv"
)

# Zip everything
shutil.make_archive(
    "StockVision_Artifacts",
    "zip",
    ".",
    "StockVision_Artifacts"
)

print("All files saved successfully!")

In [ ]:
from google.colab import files

files.download("StockVision_Artifacts.zip")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

zip_path = "/content/StockVision_GUI.zip"
extract_path = "/content/StockVision_GUI"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted successfully!")
print("Contents:")

for root, dirs, files in os.walk(extract_path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
%cd /content/StockVision_GUI/StockVision_AI

!pip install -r requirements.txt

In [ ]:
import sys

print("Python version:", sys.version)

try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
except Exception as e:
    print("TensorFlow error:", e)

In [ ]:
from pathlib import Path

requirements = Path("requirements.txt").read_text().splitlines()

requirements = [
    line for line in requirements
    if not line.strip().lower().startswith("tensorflow-cpu")
]

Path("requirements_colab.txt").write_text(
    "\n".join(requirements)
)

!pip install -r requirements_colab.txt

In [ ]:
import os

print("ZIP exists:", os.path.exists("/content/StockVision_GUI.zip"))
print("Project folder exists:", os.path.exists("/content/StockVision_GUI"))

In [ ]:
import importlib

packages = [
    "tensorflow",
    "streamlit",
    "numpy",
    "pandas",
    "sklearn",
    "plotly",
    "joblib"
]

for package in packages:
    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "Installed")
        print(f"✅ {package}: {version}")
    except Exception as e:
        print(f"❌ {package}: Not installed")

In [ ]:
!pip install tensorflow-cpu==2.21.0

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

In [ ]:
import os

project_path = "/content/StockVision_GUI/StockVision_AI"

print("Project folder:", os.path.exists(project_path))

for root, dirs, files in os.walk(project_path):
    for file in files:
        print("📄", os.path.relpath(
            os.path.join(root, file), project_path
        ))

In [ ]:
import os
import tensorflow as tf
import joblib

BASE_PATH = "/content/StockVision_GUI/StockVision_AI"

model = tf.keras.models.load_model(
    os.path.join(BASE_PATH, "artifacts/stock_model.keras")
)

scaler = joblib.load(
    os.path.join(BASE_PATH, "artifacts/scaler.pkl")
)

feature_columns = joblib.load(
    os.path.join(BASE_PATH, "artifacts/feature_columns.pkl")
)

print("✅ Model loaded successfully!")
print("✅ Scaler loaded successfully!")
print("Number of features:", len(feature_columns))
print("Model input shape:", model.input_shape)

In [ ]:
%cd /content/StockVision_GUI/StockVision_AI

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

import time
from google.colab.output import eval_js

time.sleep(10)

url = eval_js("google.colab.kernel.proxyPort(8501)")
print("StockVision AI Dashboard:")
print(url)

In [ ]:
!tail -n 40 /content/streamlit.log

In [ ]:
%cd /content/StockVision_GUI/StockVision_AI

!streamlit run app.py \
    --server.port 8502 \
    --server.address 0.0.0.0 \
    --server.enableCORS=false \
    --server.enableXsrfProtection=false \
    > /content/streamlit2.log 2>&1 &

import time
from google.colab.output import eval_js

time.sleep(10)

url = eval_js("google.colab.kernel.proxyPort(8502)")

print("StockVision AI Dashboard:")
print(url)

In [ ]:
import shutil
from google.colab import files

# Create a final ZIP of the project
shutil.make_archive(
    "/content/StockVision_AI_Final",
    "zip",
    "/content/StockVision_GUI"
)

# Download it to your computer
files.download("/content/StockVision_AI_Final.zip")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile
import os

zip_path = "/content/StockVision_AI_Final.zip"
extract_path = "/content/StockVision_GUI"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Project extracted successfully!")

for root, dirs, files in os.walk(extract_path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import tensorflow as tf
import streamlit
import joblib

print("TensorFlow:", tf.__version__)
print("Streamlit:", streamlit.__version__)
print("All packages imported successfully!")

In [ ]:
%cd /content/StockVision_GUI/StockVision_AI

!pip install -r requirements_colab.txt

In [ ]:
%cd /content/StockVision_GUI/StockVision_AI

!python -c "import tensorflow, streamlit, numpy, pandas, plotly, joblib; print('ALL LIBRARIES READY')"

In [ ]:
!pip install --upgrade numpy==2.1.3

In [ ]:
import os

project_path = "/content/StockVision_GUI/StockVision_AI"

if os.path.exists(project_path):
    print("PROJECT FILES FOUND")
    print(os.listdir(project_path))
else:
    print("PROJECT FILES NOT FOUND")

In [ ]:
import tensorflow
import streamlit
import numpy
import pandas
import plotly
import joblib

print("ALL LIBRARIES READY")

In [ ]:
%cd /content/StockVision_GUI/StockVision_AI

!python -c "import tensorflow, streamlit, numpy, pandas, plotly, joblib; print('ALL LIBRARIES READY')"

In [ ]:
%cd /content/StockVision_GUI/StockVision_AI

!streamlit run app.py \
    --server.port 8502 \
    --server.address 0.0.0.0 \
    --server.enableCORS=false \
    --server.enableXsrfProtection=false \
    --server.headless=true \
    > /content/streamlit.log 2>&1 &

In [ ]:
!tail -n 20 /content/streamlit.log

In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8502)")
print("StockVision AI Dashboard:")
print(url)

In [ ]:
app_path = "/content/StockVision_GUI/StockVision_AI/app.py"

with open(app_path, "r", encoding="utf-8") as f:
    code = f.read()

print(code)

In [ ]:
%%writefile /content/StockVision_GUI/StockVision_AI/StockVision_Model_Tester.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import plotly.express as px
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# =========================
# Configuration
# =========================

st.set_page_config(
    page_title="StockVision Model Tester",
    page_icon="📈",
    layout="wide"
)

BASE_DIR = Path(__file__).parent
ARTIFACTS = BASE_DIR / "artifacts"

SEQ_LEN = 14

RATIO_FEATURES = [
    "Trend_Velocity",
    "Trend_Accel",
    "Trend_Jerk",
    "Dist_Smooth",
    "Momentum_13D",
    "EPU_Shock",
    "GPR_Shock",
    "Macro_Crisis_Index"
]


# =========================
# Load Model and Artifacts
# =========================

@st.cache_resource
def load_assets():

    model = tf.keras.models.load_model(
        ARTIFACTS / "stock_model.keras"
    )

    scaler = joblib.load(
        ARTIFACTS / "scaler.pkl"
    )

    feature_columns = joblib.load(
        ARTIFACTS / "feature_columns.pkl"
    )

    return model, scaler, feature_columns


# =========================
# Feature Engineering
# =========================

def prepare_data(df, feature_columns):

    df = df.copy()

    if "Close" not in df.columns:
        raise ValueError("CSV must contain a Close column.")

    # Moving averages
    df["MA_14"] = df["Close"].rolling(14).mean()

    df["Smooth_Trend"] = (
        df["MA_14"].rolling(5).mean()
    )

    # Trend features
    df["Trend_Velocity"] = (
        df["Smooth_Trend"].pct_change()
    )

    df["Trend_Accel"] = (
        df["Trend_Velocity"].diff()
    )

    df["Trend_Jerk"] = (
        df["Trend_Accel"].diff()
    )

    df["Dist_Smooth"] = (
        (df["Close"] - df["Smooth_Trend"]) /
        (df["Smooth_Trend"] + 1e-8)
    )

    df["Momentum_13D"] = (
        df["Close"].pct_change(13)
    )

    # Macro features
    if "EPU" in df.columns and "GPR" in df.columns:

        epu_mean = df["EPU"].rolling(90).mean()
        gpr_mean = df["GPR"].rolling(90).mean()

        df["EPU_Shock"] = (
            (df["EPU"] - epu_mean) /
            (epu_mean + 1e-8)
        )

        df["GPR_Shock"] = (
            (df["GPR"] - gpr_mean) /
            (gpr_mean + 1e-8)
        )

        df["Macro_Crisis_Index"] = (
            df["EPU_Shock"] + df["GPR_Shock"]
        ) / 2

    # Actual labels: next day's trend
    actual = np.where(
        df["Smooth_Trend"].shift(-1) >
        df["Smooth_Trend"],
        1,
        0
    ).astype(float)

    actual[-1] = np.nan

    # Exclude target columns from model inputs
    excluded = [
        "Target_Next_Close",
        "Target_Direction",
        "Target_Trend"
    ]

    candidates = [
        c for c in df.columns
        if c not in excluded
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    # Rolling normalization
    for col in candidates:

        if col in RATIO_FEATURES:

            rolling_std = df[col].rolling(30).std()

            df[col] = (
                df[col] / (rolling_std + 1e-8)
            ).clip(-4, 4)

        else:

            rolling_mean = df[col].rolling(30).mean()
            rolling_std = df[col].rolling(30).std()

            df[col] = (
                (df[col] - rolling_mean) /
                (rolling_std + 1e-8)
            ).clip(-4, 4)

    missing = [
        c for c in feature_columns
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required features: {missing}"
        )

    X = df[feature_columns].copy()

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)

    return X, pd.Series(actual, index=df.index)


# =========================
# Model Prediction
# =========================

def predict_model(X, actual, model, scaler):

    if len(X) < SEQ_LEN:
        raise ValueError(
            f"CSV needs at least {SEQ_LEN} rows."
        )

    scaled = scaler.transform(
        X.to_numpy(dtype=np.float32)
    )

    scaled = np.clip(scaled, -4, 4)

    sequences = []

    for i in range(SEQ_LEN - 1, len(scaled)):

        sequences.append(
            scaled[i - SEQ_LEN + 1:i + 1]
        )

    sequences = np.array(
        sequences,
        dtype=np.float32
    )

    probabilities = model.predict(
        sequences,
        verbose=0
    ).reshape(-1)

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    positions = range(
        SEQ_LEN - 1,
        len(X)
    )

    results = pd.DataFrame({

        "Date": [X.index[i] for i in positions],

        "Actual": [
            actual.iloc[i]
            for i in positions
        ],

        "Predicted": predictions,

        "Probability_Up": probabilities

    })

    return results


# =========================
# Metrics
# =========================

def show_metrics(results):

    valid = results.dropna(
        subset=["Actual"]
    )

    if len(valid) == 0:

        st.warning(
            "No actual labels available. "
            "Metrics cannot be calculated."
        )

        return

    y_true = valid["Actual"].astype(int)
    y_pred = valid["Predicted"].astype(int)

    c1, c2, c3, c4 = st.columns(4)

    c1.metric(
        "Accuracy",
        f"{accuracy_score(y_true, y_pred):.4f}"
    )

    c2.metric(
        "Precision",
        f"{precision_score(y_true, y_pred, zero_division=0):.4f}"
    )

    c3.metric(
        "Recall",
        f"{recall_score(y_true, y_pred, zero_division=0):.4f}"
    )

    c4.metric(
        "F1 Score",
        f"{f1_score(y_true, y_pred, zero_division=0):.4f}"
    )

    # Confusion Matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    fig = px.imshow(
        cm,
        text_auto=True,
        x=["Predicted 0", "Predicted 1"],
        y=["Actual 0", "Actual 1"],
        title="Confusion Matrix",
        color_continuous_scale="Blues"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

    # Classification Report
    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Down/Stable", "Up"],
        output_dict=True,
        zero_division=0
    )

    st.subheader("Classification Report")

    st.dataframe(
        pd.DataFrame(report).transpose(),
        use_container_width=True
    )


# =========================
# Main GUI
# =========================

def main():

    st.title("📈 StockVision AI")
    st.subheader("Standalone Model Testing Interface")

    st.write(
        "Upload a CSV file to test the saved StockVision model."
    )

    try:

        model, scaler, feature_columns = load_assets()

    except Exception as e:

        st.error(f"Model loading error: {e}")
        st.stop()

    st.sidebar.header("Model Information")

    st.sidebar.write(
        f"Features: {len(feature_columns)}"
    )

    st.sidebar.write(
        f"Sequence Length: {SEQ_LEN}"
    )

    uploaded_file = st.file_uploader(
        "Upload CSV Dataset",
        type=["csv"]
    )

    if uploaded_file is None:

        st.info("Please upload a CSV file.")
        return

    try:

        df = pd.read_csv(uploaded_file)

        if "Date" in df.columns:

            df["Date"] = pd.to_datetime(
                df["Date"],
                errors="coerce"
            )

            df = df.dropna(
                subset=["Date"]
            ).set_index("Date")

        df = df.sort_index()

        df = df.apply(
            pd.to_numeric,
            errors="coerce"
        )

        df = df.replace(
            [np.inf, -np.inf],
            np.nan
        ).ffill().bfill().fillna(0)

        st.write(
            f"Dataset shape: {df.shape}"
        )

        st.dataframe(
            df.head(),
            use_container_width=True
        )

        if st.button(
            "Run Model Testing",
            type="primary"
        ):

            with st.spinner(
                "Running predictions..."
            ):

                X, actual = prepare_data(
                    df,
                    feature_columns
                )

                results = predict_model(
                    X,
                    actual,
                    model,
                    scaler
                )

            st.success(
                "Predictions completed successfully!"
            )

            st.warning(
                "Important: If this CSV overlaps with training data, "
                "the resulting metrics are not an independent test evaluation."
            )

            st.header("Evaluation Metrics")

            show_metrics(results)

            st.header("Actual vs Predicted")

            st.dataframe(
                results,
                use_container_width=True
            )

            st.header("Prediction Probabilities")

            fig = px.line(
                results,
                x="Date",
                y="Probability_Up",
                title="Probability of Up Trend"
            )

            st.plotly_chart(
                fig,
                use_container_width=True
            )

            # Download results
            csv = results.to_csv(
                index=False
            ).encode("utf-8-sig")

            st.download_button(
                "Download Results CSV",
                data=csv,
                file_name="StockVision_Test_Results.csv",
                mime="text/csv"
            )

    except Exception as e:

        st.error(
            f"Testing error: {e}"
        )


if __name__ == "__main__":

    main()

In [ ]:
%cd /content/StockVision_GUI/StockVision_AI

!streamlit run StockVision_Model_Tester.py \
    --server.port 8503 \
    --server.address 0.0.0.0 \
    --server.enableCORS=false \
    --server.enableXsrfProtection=false \
    --server.headless=true \
    > /content/model_tester.log 2>&1 &

In [ ]:
from google.colab.output import eval_js
import time

time.sleep(5)

url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

In [ ]:
!echo "===== STREAMLIT PROCESS ====="
!ps aux | grep '[s]treamlit'

!echo "===== LAST LOG MESSAGES ====="
!tail -30 /content/model_tester.log

In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8503)")
print("StockVision Model Tester:", url)

In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8502)")
print("Original Dashboard:", url)

!tail -40 /content/streamlit.log

In [ ]:
import requests

for port in [8502, 8503]:
    try:
        response = requests.get(
            f"http://127.0.0.1:{port}/_stcore/health",
            timeout=10
        )
        print(f"Port {port}: {response.status_code} - {response.text}")
    except Exception as e:
        print(f"Port {port}: ERROR - {e}")